# Week 8 - Day 1

# Text Preprocessing for NLP: Tokenization, Stopwords, Stemming, and TF-IDF

## A Beginner-Friendly, Fully Explained Walkthrough

Everything we've built so far - loan approvals, digit images, coloured shapes - was already made of numbers. Text is different: `"This movie was fantastic!"` isn't a number a model can multiply and add. Before any Machine Learning model can work with text, we have to turn words into numbers, in a way that actually preserves their meaning.

Today is entirely about that conversion process, called **text preprocessing**. We will not rush. Every new code cell is followed by a plain-English, line-by-line explanation.

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain why raw text can't be fed directly into a Machine Learning model.
- Explain and perform tokenization - splitting text into individual words.
- Explain why stopwords are removed, and remove them yourself.
- Explain the difference between stemming and lemmatization, and use both.
- Explain the Bag-of-Words (BoW) representation, and build one by hand and with scikit-learn.
- Explain TF-IDF and why it usually works better than plain word counts.
- Build a full, reusable text-cleaning pipeline function.
- Apply that pipeline to a small dataset of real-looking reviews and export the cleaned result.

## Why This Lesson Matters

Imagine reading through hundreds of customer reviews for a Nigerian delivery app - some in full sentences, some full of typos, some with "the," "is," and "and" scattered everywhere, some saying the same thing with completely different words ("fast delivery" vs "delivered quickly"). A human skims past all that noise instinctively. A Machine Learning model can't - it needs the text turned into clean, consistent, numeric building blocks first. Today's lesson is exactly that translation step, done carefully, one technique at a time.

## Recap: Where We've Been

| Week | Data Type | Model |
|---|---|---|
| Week 5-6 | Tabular data / Images | Logistic Regression, Decision Trees, ANN |
| Week 7 | Images | CNN |
| **Week 8** | **Text** | **(today: preprocessing) -> tomorrow: sentiment classifier** |

One idea carries over directly from Week 7: just as a CNN had to handle a shape appearing *anywhere* in an image, an NLP model has to handle the *same word* appearing in different forms ("running," "ran," "runs") or in different positions in a sentence. Today's preprocessing steps are how we tame that variability before any model ever sees the text.

# Part 1 - Why Machines Can't Read Text Directly

A Neural Network, at its core, only ever does one thing: multiply numbers by weights and add them up (remember the Perceptron from Week 6 Day 1). It has no built-in understanding of grammar, spelling, or meaning.

Consider these two sentences:

> "The jollof rice was absolutely delicious!"
> "the Jollof RICE was absolutely delicious"

To a human, these mean the exact same thing. To a computer comparing raw text character-by-character, they look almost entirely different - different capitalization, different punctuation. Before we can let a model learn from text, we need to strip away this kind of meaningless variation, while keeping the actual meaning intact.

# Part 2 - A Small Dataset of Reviews

Let's build a small, realistic dataset of product/movie-style reviews to practice on - similar in spirit to the movie-review example in the lesson plan, written in a mix of styles (some short, some long, some with typos) the way real reviews actually look.

In [1]:
import pandas as pd

reviews_data = {
    "review": [
        "This movie was absolutely fantastic! The acting was superb.",
        "I did not enjoy this film at all, it was too slow and boring.",
        "An amazing storyline with great characters, I loved every minute.",
        "Terrible movie, waste of my time and money.",
        "The plot was confusing but the visuals were stunning.",
        "Best Nollywood film I have seen this year, highly recommend!",
        "I fell asleep halfway through, it was that dull.",
        "Great acting, great music, great everything. A masterpiece!",
        "Not worth watching, the story made no sense to me.",
        "One of the worst films I've ever sat through, truly awful.",
        "The cinematography was breathtaking and the story touched my heart.",
        "Mediocre at best, I expected so much more from this director.",
        "A brilliant, heartwarming film that I will watch again and again.",
        "I regret spending money on this ticket, complete disappointment.",
        "The comedy scenes had the whole cinema laughing out loud!",
        "Poorly written script with weak character development.",
        "Such a beautiful, emotional journey, I cried at the end.",
        "The special effects looked cheap and the acting felt forced.",
        "A must-watch! Everyone should see this incredible movie.",
        "I would not recommend this film to anyone, honestly boring.",
        "The lead actress gave an outstanding, unforgettable performance.",
        "Predictable and dull, nothing new or exciting happened.",
        "This film exceeded all my expectations, absolutely wonderful.",
        "The pacing was off and the ending felt rushed and unsatisfying.",
        "A joy to watch from start to finish, wonderfully directed.",
        "I couldn't connect with any of the characters at all.",
        "Powerful storytelling with a message that stays with you.",
        "Way too long and the plot dragged on forever.",
        "Funny, touching, and beautifully shot, I loved it!",
        "Disappointing sequel, it lost the charm of the original.",
    ],
    "sentiment": [
        "positive", "negative", "positive", "negative", "neutral",
        "positive", "negative", "positive", "negative", "negative",
        "positive", "negative", "positive", "negative", "positive",
        "negative", "positive", "negative", "positive", "negative",
        "positive", "negative", "positive", "negative", "positive",
        "negative", "positive", "negative", "positive", "negative",
    ]
}

reviews_df = pd.DataFrame(reviews_data)
print("Number of reviews:", len(reviews_df))
reviews_df.head(10)

Number of reviews: 30


,review,sentiment
0,This movie was absolutely fantastic! The actin...,positive
1,"I did not enjoy this film at all, it was too s...",negative
2,"An amazing storyline with great characters, I ...",positive
3,"Terrible movie, waste of my time and money.",negative
4,The plot was confusing but the visuals were st...,neutral
5,"Best Nollywood film I have seen this year, hig...",positive
6,"I fell asleep halfway through, it was that dull.",negative
7,"Great acting, great music, great everything. A...",positive
8,"Not worth watching, the story made no sense to...",negative
9,"One of the worst films I've ever sat through, ...",negative


### Line-by-Line Breakdown

- **`reviews_data = {...}`** - We hand-build a Python dictionary with two keys: `"review"` (the raw text of each review) and `"sentiment"` (a label - `positive`, `negative`, or `neutral` - which we'll use for our sentiment classifier tomorrow in Day 2). Each key maps to a list of 30 matching values, in the same order.
- **`reviews_df = pd.DataFrame(reviews_data)`** - Converts that dictionary into a proper table, exactly the same pattern we've used since Week 3 - each dictionary key becomes a column.
- **`print(...)`** and **`reviews_df.head(10)`** - Confirms we have 30 reviews, and previews the first 10 rows.

Notice how messy real writing already is, even though we wrote this ourselves: inconsistent capitalization, some short sentences, some long ones, punctuation like `!` and `'`. This is completely normal, and exactly what preprocessing is for.

# Part 3 - Tokenization

**Tokenization** means splitting a piece of text into smaller units - usually individual words, called **tokens**. It's the very first step of almost every NLP pipeline, because you can't clean or count "words" until you've decided exactly where one word ends and the next begins.

## The Simplest Possible Tokenizer

Let's write our own, by hand, before using a ready-made tool - so we understand exactly what's happening underneath.

In [2]:
def simple_tokenize(text):
    text = text.lower()
    tokens = text.split()
    return tokens

sample_review = reviews_df.loc[0, "review"]
print("Original text:", sample_review)
print("Tokens:", simple_tokenize(sample_review))

Original text: This movie was absolutely fantastic! The acting was superb.
Tokens: ['this', 'movie', 'was', 'absolutely', 'fantastic!', 'the', 'acting', 'was', 'superb.']


### Line-by-Line Breakdown

- **`def simple_tokenize(text):`** - Defines our own function, taking one piece of text as input.
- **`text = text.lower()`** - Converts all letters to lowercase. This matters because, to a computer, `"Movie"` and `"movie"` are two completely different strings unless we normalize the case first - we want the model to treat them as the same word.
- **`tokens = text.split()`** - Python's built-in `.split()`, called with no arguments, splits a string wherever it finds *any* whitespace (spaces, tabs, newlines), and returns a list of the pieces in between.
- **`return tokens`** - Hands back that list of words.
- **`reviews_df.loc[0, "review"]`** - `.loc[row, column]` grabs one specific value by its row and column label - here, the review text in row `0`.

Try reading the printed output: notice that `"fantastic!"` came out as one token *including* the exclamation mark, since `.split()` only splits on whitespace, not punctuation. That's a real limitation of this simple approach - which is exactly why real-world tools are more careful.

## A More Careful Tokenizer, Using NLTK

**NLTK** (Natural Language Toolkit) is a widely-used Python library built specifically for text processing. Its tokenizer understands punctuation properly - it knows `"fantastic!"` should become two tokens: `"fantastic"` and `"!"`, and that `"I've"` should become `"I"`, `"'ve"`.

### Installation Guide

**Step 1 - Install the library.**
```bash
pip install nltk
```

**Step 2 - Download NLTK's language data.** NLTK's tokenizer, stopword list, and lemmatizer all rely on small downloaded data files (not code) - this is a one-time setup step.
```python
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
```

**Step 3 - Verify it worked** by running the cell below.

In [3]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("NLTK setup successful.")
print("Tokens:", word_tokenize(sample_review.lower()))

NLTK setup successful.
Tokens: ['this', 'movie', 'was', 'absolutely', 'fantastic', '!', 'the', 'acting', 'was', 'superb', '.']


### Line-by-Line Breakdown

- **`nltk.download("punkt", quiet=True)`** - Downloads NLTK's pre-trained sentence/word-splitting model, if it isn't already on your computer (`quiet=True` just hides the progress messages so our output stays tidy). `"punkt_tab"` is a companion data file that newer NLTK versions also need for the same tokenizer.
- **`from nltk.tokenize import word_tokenize`** - Imports the ready-made tokenizer function.
- **`word_tokenize(sample_review.lower())`** - Runs it on our lowercased sample review.

Compare this output to our hand-written `simple_tokenize()` result above: notice `"fantastic!"` is now correctly split into `"fantastic"` and `"!"` as two separate tokens.

# Part 4 - Stopword Removal

**Stopwords** are extremely common words - `"the"`, `"is"`, `"a"`, `"and"`, `"was"` - that appear constantly in almost every sentence, but carry very little meaning on their own. Removing them lets a model focus its attention on the words that actually distinguish one piece of text from another.

In [4]:
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)

english_stopwords = set(stopwords.words("english"))

print("Number of English stopwords:", len(english_stopwords))
print("A few examples:", list(english_stopwords)[:15])

Number of English stopwords: 198
A few examples: ['its', 'against', 'how', 'itself', 'on', 'our', "she's", 'at', "he's", "couldn't", 'themselves', "you're", 'both', "he'd", 'wouldn']


### Line-by-Line Breakdown

- **`nltk.download("stopwords", quiet=True)`** - Downloads NLTK's built-in list of common stopwords, one time.
- **`stopwords.words("english")`** - Returns that list as plain Python strings, specifically for English (NLTK has stopword lists for many other languages too).
- **`set(...)`** - Converts the list into a Python `set`. Sets check "is this item inside?" much faster than lists do, which matters because we're about to check *every single token* against this list - using a set instead of a list makes that check far more efficient.

In [5]:
def remove_stopwords(tokens, stopword_set):
    return [word for word in tokens if word not in stopword_set]

tokens = word_tokenize(sample_review.lower())
filtered_tokens = remove_stopwords(tokens, english_stopwords)

print("Before removing stopwords:", tokens)
print("After removing stopwords: ", filtered_tokens)

Before removing stopwords: ['this', 'movie', 'was', 'absolutely', 'fantastic', '!', 'the', 'acting', 'was', 'superb', '.']
After removing stopwords:  ['movie', 'absolutely', 'fantastic', '!', 'acting', 'superb', '.']


### Line-by-Line Breakdown

- **`[word for word in tokens if word not in stopword_set]`** - This is a **list comprehension**: a compact way of writing a loop that builds a new list. In plain English: "go through every `word` in `tokens`, and keep it in the new list only `if` that `word` is `not in` our stopword set." Anything that *is* a stopword gets silently left out.
- **Comparing the before/after output** - Notice words like `"was"`, `"the"` disappear, while meaningful words like `"fantastic"`, `"acting"`, `"superb"` remain - exactly the effect we want.

### A Word of Caution

Stopword removal is a genuinely useful default, but it isn't always harmless. Consider the sentence "this movie was **not** good" - `"not"` is often included in stopword lists, and removing it would flip the entire meaning of the sentence for a model relying only on remaining words! For today's lesson we'll follow the standard approach, but it's worth knowing this trade-off exists - more advanced NLP techniques (which use word order and context) handle this much better than simple stopword removal can.

# Part 5 - Stemming and Lemmatization

Words like `"run"`, `"running"`, and `"ran"` are all different forms of the same underlying idea. Without some way to unify them, a model would treat all three as completely unrelated words - wasting the pattern that they actually mean almost the same thing.

## Stemming: Fast but Rough

**Stemming** chops words down to a rough "stem" using simple, mechanical rules - it doesn't understand grammar, it just trims common endings.

In [6]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

example_words = ["running", "runs", "ran", "easily", "fairness", "movies", "better"]

for word in example_words:
    print(f"{word:10} -> {stemmer.stem(word)}")

running    -> run
runs       -> run
ran        -> ran
easily     -> easili
fairness   -> fair
movies     -> movi
better     -> better


### Line-by-Line Breakdown

- **`stemmer = PorterStemmer()`** - Creates a stemmer object using the **Porter algorithm**, a well-known, rule-based stemming method (chop off `"-ing"`, `"-s"`, `"-ed"`, and other common endings, following a specific set of rules).
- **`stemmer.stem(word)`** - Runs the algorithm on a single word.
- **`f"{word:10} -> {stemmer.stem(word)}"`** - An f-string with column-width formatting: `{word:10}` pads `word` out to 10 characters wide, so the arrows line up neatly in the printed output.

**Notice the rough edges:** `"running"` and `"runs"` both correctly become `"run"`, but `"better"` stays as `"better"` (a stemmer has no idea it should relate to `"good"`), and `"fairness"` becomes `"fair"` while `"easily"` becomes `"easili"` - not even a real English word! Stemming is fast, but it can produce nonsense.

## Lemmatization: Slower but Grammatically Correct

**Lemmatization** instead reduces a word to its proper dictionary form (called a "lemma"), using actual linguistic knowledge rather than blunt rule-chopping.

In [7]:
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet", quiet=True)
lemmatizer = WordNetLemmatizer()

for word in example_words:
    print(f"{word:10} -> {lemmatizer.lemmatize(word, pos='v')}")

running    -> run
runs       -> run
ran        -> run
easily     -> easily
fairness   -> fairness
movies     -> movies
better     -> better


### Line-by-Line Breakdown

- **`nltk.download("wordnet", quiet=True)`** - WordNet is a large, structured English dictionary that NLTK's lemmatizer relies on to know real word relationships (e.g. that "running" comes from "run").
- **`WordNetLemmatizer()`** - Creates the lemmatizer object.
- **`lemmatizer.lemmatize(word, pos='v')`** - Looks up the correct dictionary form for `word`. The `pos='v'` argument tells it to assume the word is being used as a **v**erb - lemmatization needs to know a word's part of speech (verb, noun, adjective) to lemmatize it correctly, since the same spelling can lemmatize differently depending on its grammatical role.

**Compare the results:** `"running"` still correctly becomes `"run"`, and this time `"easily"` and `"fairness"` are left as real, valid words (since they aren't verbs, and we told it to only apply verb-style rules) rather than being mangled.

### Stemming vs Lemmatization: When to Use Which

| | Stemming | Lemmatization |
|---|---|---|
| Speed | Fast | Slower |
| Accuracy | Rough, can produce non-words | Grammatically correct, real words |
| Needs a dictionary? | No | Yes (WordNet) |
| Good for | Quick prototypes, search engines | Tasks where meaning/readability matters |

For today's dataset (a small handful of reviews), either would work fine - but lemmatization tends to be the better default choice for tasks like sentiment analysis (tomorrow's lesson), where preserving accurate word meaning genuinely matters.

# Part 6 - Bag-of-Words (BoW)

Now that we can clean text down to a list of meaningful word-tokens, how do we turn that into actual **numbers** a model can use? The simplest approach is **Bag-of-Words**: count how many times each word appears, and ignore word order entirely (hence "bag" - like dumping all the words into a bag, with no sense of sequence).

## Building One by Hand

Let's do this manually first, on just 3 short example sentences, so the idea is completely concrete before we use a ready-made tool.

In [8]:
from collections import Counter

mini_corpus = [
    "the movie was great",
    "the movie was boring",
    "great acting great story",
]

tokenized_corpus = [sentence.split() for sentence in mini_corpus]
print("Tokenized:", tokenized_corpus)

vocabulary = sorted(set(word for sentence in tokenized_corpus for word in sentence))
print("\nVocabulary (every unique word):", vocabulary)

for sentence_tokens in tokenized_corpus:
    counts = Counter(sentence_tokens)
    row = [counts.get(word, 0) for word in vocabulary]
    print(sentence_tokens, "->", row)

Tokenized: [['the', 'movie', 'was', 'great'], ['the', 'movie', 'was', 'boring'], ['great', 'acting', 'great', 'story']]

Vocabulary (every unique word): ['acting', 'boring', 'great', 'movie', 'story', 'the', 'was']
['the', 'movie', 'was', 'great'] -> [0, 0, 1, 1, 0, 1, 1]
['the', 'movie', 'was', 'boring'] -> [0, 1, 0, 1, 0, 1, 1]
['great', 'acting', 'great', 'story'] -> [1, 0, 2, 0, 1, 0, 0]


### Line-by-Line Breakdown

- **`from collections import Counter`** - `Counter` is a specialised Python dictionary built specifically for counting things - handing it a list gives you back how many times each item appeared.
- **`mini_corpus = [...]`** - "Corpus" is the standard NLP term for "a collection of text documents." We use 3 tiny sentences here to keep the example small enough to trace by hand.
- **`tokenized_corpus = [sentence.split() for sentence in mini_corpus]`** - A list comprehension (from Part 4) that tokenizes every sentence in the corpus at once, producing a list of token-lists.
- **`vocabulary = sorted(set(word for sentence in tokenized_corpus for word in sentence))`** - This builds the full **vocabulary**: every unique word that appears *anywhere* across all 3 sentences. The nested loop (`for sentence in tokenized_corpus for word in sentence`) walks through every sentence and every word inside it; wrapping it in `set(...)` keeps only unique words; `sorted(...)` puts them in a fixed, predictable alphabetical order (important, since this order determines what each number in our final output represents).
- **`Counter(sentence_tokens)`** - Counts how many times each word appears in *this specific sentence*.
- **`[counts.get(word, 0) for word in vocabulary]`** - For every word in our full vocabulary (in that same fixed order), look up how many times it appeared in this sentence (`counts.get(word, 0)` returns `0` if the word isn't in this sentence at all, rather than raising an error).

**Reading the output:** each sentence becomes a row of numbers, the same length as our vocabulary - one number per vocabulary word, showing how many times it appeared in that sentence. This is a Bag-of-Words representation: pure counts, no sense of word order or sentence structure at all.

## The Same Idea, Using scikit-learn

Doing this by hand for a large vocabulary and many documents would get unwieldy fast - scikit-learn's `CountVectorizer` does exactly the same thing, but efficiently and with helpful extra options.

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(mini_corpus)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=mini_corpus
)
bow_df

,acting,boring,great,movie,story,the,was
the movie was great,0,0,1,1,0,1,1
the movie was boring,0,1,0,1,0,1,1
great acting great story,1,0,2,0,1,0,0


### Line-by-Line Breakdown

- **`CountVectorizer()`** - Creates a Bag-of-Words converter object, with its default settings (automatically lowercases text and handles basic tokenization for us).
- **`vectorizer.fit_transform(mini_corpus)`** - Two things happen in one call: **`fit`** learns the vocabulary from our corpus (same idea as building `vocabulary` by hand above), and **`transform`** converts every sentence into its count-vector, exactly like our hand-built rows.
- **`bow_matrix.toarray()`** - The result comes back in a specialised, memory-efficient "sparse matrix" format (since real vocabularies are often huge, and most counts are 0). `.toarray()` converts it into a regular, easy-to-read grid for display.
- **`vectorizer.get_feature_names_out()`** - Returns the vocabulary words, in the exact order the columns represent - we use this as our column names.
- **`index=mini_corpus`** - Uses the original sentences themselves as row labels, so the table is easy to read.

Compare this table's numbers to what we calculated by hand above - they should match exactly, just presented more conveniently.

# Part 7 - TF-IDF: A Smarter Word-Weighting Scheme

Plain word counts have a problem: a word like `"movie"` might appear in *every single review* in our dataset, contributing nothing useful for telling reviews apart, yet Bag-of-Words would give it a large count in every row regardless. **TF-IDF** (Term Frequency - Inverse Document Frequency) fixes this by lowering the weight of words that appear in *many* documents, and boosting the weight of words that are more distinctive.

### The Idea, in Plain Language

- **Term Frequency (TF)** - how often a word appears *within this one document* (similar to our Bag-of-Words count).
- **Inverse Document Frequency (IDF)** - a penalty based on how common that word is *across the entire corpus*. A word appearing in almost every document gets a small IDF value (low importance); a word appearing in only a few documents gets a large IDF value (high importance, since it's more distinctive).
- **TF-IDF = TF × IDF** - multiplying these together gives words that are both *frequent in this document* **and** *rare across the whole corpus* the highest scores - exactly the words that best characterise what makes this particular piece of text distinctive.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(mini_corpus)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=mini_corpus
)
tfidf_df.round(3)

,acting,boring,great,movie,story,the,was
the movie was great,0.000,0.000,0.500,0.50,0.000,0.50,0.50
the movie was boring,0.000,0.605,0.000,0.46,0.000,0.46,0.46
great acting great story,0.481,0.000,0.732,0.00,0.481,0.00,0.00


### Line-by-Line Breakdown

- **`TfidfVectorizer()`** - Same idea as `CountVectorizer`, but computes TF-IDF weights instead of raw counts.
- **`.fit_transform(mini_corpus)`** - Same two-step process as before: learns the vocabulary and IDF weights from the corpus, then converts every sentence into its TF-IDF vector.
- **`.round(3)`** - Rounds every value to 3 decimal places purely for a tidier display (TF-IDF scores are decimals, not whole numbers, unlike Bag-of-Words counts).

**Compare this table to the Bag-of-Words table from Part 6.** Notice the word `"the"` and `"movie"` - both appear in 2 out of 3 sentences, so they get relatively *lower* TF-IDF scores than a word like `"boring"`, which appears in only 1 sentence and therefore gets a relatively *higher* score, marking it as more distinctive to that specific sentence.

### Why TF-IDF Usually Beats Plain Word Counts

Going back to our earlier example: a word like `"movie"` appearing in every single review tells a sentiment classifier almost nothing about whether a review is positive or negative - every review mentions "movie." A word like `"boring"` or `"fantastic"`, appearing in only a handful of reviews, is far more informative for actually distinguishing between them. TF-IDF automatically captures that intuition as a number, without us having to manually decide which words matter.

# Part 8 - Putting It All Together: A Full Cleaning Pipeline

Let's combine every technique from today into one reusable function, then apply it to our entire 30-review dataset from Part 2.

In [11]:
lemmatizer = WordNetLemmatizer()
stopword_set = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.isalpha()]
    tokens = [word for word in tokens if word not in stopword_set]
    tokens = [lemmatizer.lemmatize(word, pos="v") for word in tokens]
    return " ".join(tokens)

reviews_df["cleaned_review"] = reviews_df["review"].apply(clean_text)
reviews_df[["review", "cleaned_review"]].head(10)

,review,cleaned_review
0,This movie was absolutely fantastic! The actin...,movie absolutely fantastic act superb
1,"I did not enjoy this film at all, it was too s...",enjoy film slow bore
2,"An amazing storyline with great characters, I ...",amaze storyline great character love every minute
3,"Terrible movie, waste of my time and money.",terrible movie waste time money
4,The plot was confusing but the visuals were st...,plot confuse visuals stun
5,"Best Nollywood film I have seen this year, hig...",best nollywood film see year highly recommend
6,"I fell asleep halfway through, it was that dull.",fell asleep halfway dull
7,"Great acting, great music, great everything. A...",great act great music great everything masterp...
8,"Not worth watching, the story made no sense to...",worth watch story make sense
9,"One of the worst films I've ever sat through, ...",one worst film ever sit truly awful


### Line-by-Line Breakdown

- **`def clean_text(text):`** - One function that chains together everything from today, in order.
- **`text = text.lower()`** - Lowercases the whole review (Part 3).
- **`tokens = word_tokenize(text)`** - Splits it into tokens, punctuation-aware (Part 3).
- **`tokens = [word for word in tokens if word.isalpha()]`** - A new step: `.isalpha()` checks whether a string contains *only* letters, with no punctuation, numbers, or symbols. This list comprehension keeps only "real word" tokens, dropping standalone punctuation marks like `"!"` that tokenization split out separately.
- **`tokens = [word for word in tokens if word not in stopword_set]`** - Removes stopwords (Part 4).
- **`tokens = [lemmatizer.lemmatize(word, pos="v") for word in tokens]`** - Lemmatizes every remaining word (Part 5).
- **`return " ".join(tokens)`** - `" ".join(list)` glues a list of strings back together into one string, with a single space between each item - turning our cleaned token list back into a readable sentence.
- **`reviews_df["review"].apply(clean_text)`** - `.apply()` runs our function on *every single value* in the `review` column, one at a time, and collects all the results. We store them in a brand-new column, `cleaned_review`, keeping the original `review` column untouched for comparison.

## Convert the Whole Cleaned Dataset to TF-IDF

Finally, let's turn our entire cleaned dataset into a TF-IDF matrix - ready to be handed to a classifier tomorrow.

In [12]:
final_tfidf_vectorizer = TfidfVectorizer(max_features=30)
final_tfidf_matrix = final_tfidf_vectorizer.fit_transform(reviews_df["cleaned_review"])

print("TF-IDF matrix shape:", final_tfidf_matrix.shape)
print("(", final_tfidf_matrix.shape[0], "reviews x", final_tfidf_matrix.shape[1], "vocabulary words )")

final_tfidf_df = pd.DataFrame(
    final_tfidf_matrix.toarray(),
    columns=final_tfidf_vectorizer.get_feature_names_out()
)
final_tfidf_df.head()

TF-IDF matrix shape: (30, 30)
( 30 reviews x 30 vocabulary words )


,absolutely,act,asleep,awful,beautiful,beautifully,best,bore,character,cheap,...,great,love,money,movie,plot,recommend,see,story,touch,watch
0,0.612005,0.559218,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.559218,0.000000,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.801549,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.542694,0.0,...,0.593921,0.593921,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.738226,0.674553,0.000000,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.665498,0.0,0.0,0.0,0.0,0.0


### Line-by-Line Breakdown

- **`TfidfVectorizer(max_features=30)`** - Same tool from Part 7, but this time with a limit: `max_features=30` tells it to keep only the 30 most frequent words across the whole dataset as our vocabulary, instead of every single word that appears even once. This is a common real-world practice - it keeps the resulting table a manageable size and drops extremely rare, one-off words that are unlikely to generalise well.
- **`.fit_transform(reviews_df["cleaned_review"])`** - Runs on our *cleaned* text column, not the original raw reviews - this is the entire point of everything we did in Part 8's `clean_text()` function.
- **`.shape`** - Confirms the matrix's size: one row per review, one column per vocabulary word.

## Save the Cleaned Dataset

As the lesson plan's assignment asks, let's export our cleaned, tokenized dataset to a CSV file.

In [13]:
reviews_df.to_csv("cleaned_reviews.csv", index=False)
print("Saved cleaned_reviews.csv with", len(reviews_df), "rows.")
reviews_df.head()

Saved cleaned_reviews.csv with 30 rows.


,review,sentiment,cleaned_review
0,This movie was absolutely fantastic! The actin...,positive,movie absolutely fantastic act superb
1,"I did not enjoy this film at all, it was too s...",negative,enjoy film slow bore
2,"An amazing storyline with great characters, I ...",positive,amaze storyline great character love every minute
3,"Terrible movie, waste of my time and money.",negative,terrible movie waste time money
4,The plot was confusing but the visuals were st...,neutral,plot confuse visuals stun


### Line-by-Line Breakdown

- **`reviews_df.to_csv("cleaned_reviews.csv", index=False)`** - Writes our DataFrame (now containing the original `review`, its `sentiment` label, and our new `cleaned_review` column) out to a CSV file. `index=False` leaves out pandas' internal row-number index from the saved file, since it isn't meaningful data we want to keep.

# Mini Practical Activity

1. Add 5 of your own review sentences (about anything - food, music, a product) to `reviews_data` in Part 2, rebuild `reviews_df`, and re-run the full pipeline from Part 8.
2. Change `clean_text()` to use stemming (`PorterStemmer`) instead of lemmatization. Compare a few `cleaned_review` outputs before and after - which looks more readable to you?
3. Try `max_features=10` instead of `30` in Part 8's final `TfidfVectorizer`. Which 10 words made the cut? Are they the ones you'd expect to matter most for sentiment?

### Exercise 1

Using `english_stopwords` from Part 4, check whether the word `"not"` is included in NLTK's default stopword list. Find one review in our dataset where removing `"not"` could change how a human would read its sentiment.

### Exercise 2

Pick any 3 words from `final_tfidf_df`'s columns. For each, find the review with the *highest* TF-IDF score for that word, and explain in one sentence why that word being prominent there makes sense.

### Exercise 3

Explain in your own words, using today's `"movie"` vs `"boring"` example, why TF-IDF is generally more useful than plain Bag-of-Words counts for a task like sentiment classification.

# Lesson Summary

Today, you learned how to:

- Explain why raw text must be converted into clean, consistent numbers before modelling.
- Tokenize text, both by hand and using NLTK's punctuation-aware tokenizer.
- Remove stopwords to focus on meaningful words.
- Apply stemming and lemmatization, and explain the trade-offs between them.
- Build a Bag-of-Words representation by hand and with `CountVectorizer`.
- Build a TF-IDF representation with `TfidfVectorizer`, and explain why it usually outperforms plain word counts.
- Combine every step into one reusable cleaning pipeline function.
- Export a cleaned dataset, ready for tomorrow's sentiment classifier.

# Functions and Concepts Learned Today

| Item | Purpose |
|---|---|
| Tokenization | Splitting text into individual word units |
| `word_tokenize()` | NLTK's punctuation-aware tokenizer |
| Stopwords | Extremely common, low-meaning words (e.g. "the," "is") |
| `stopwords.words("english")` | NLTK's built-in English stopword list |
| Stemming | Fast, rule-based word-shortening (can produce non-words) |
| `PorterStemmer` | A classic, widely-used stemming algorithm |
| Lemmatization | Grammar-aware reduction to a word's dictionary form |
| `WordNetLemmatizer` | NLTK's dictionary-based lemmatizer |
| Bag-of-Words (BoW) | Representing text as word counts, ignoring order |
| `CountVectorizer` | scikit-learn's Bag-of-Words converter |
| TF-IDF | Weighting scheme that favours words that are frequent locally but rare globally |
| `TfidfVectorizer` | scikit-learn's TF-IDF converter |
| `.apply()` | Runs a function on every value in a pandas column |
| `" ".join(list)` | Combines a list of strings back into one string |

# Key Takeaways

✅ Machine Learning models only understand numbers - text must be converted, carefully, before any modelling can happen.

✅ Tokenization, lowercasing, and punctuation-handling are the foundation every other NLP step builds on.

✅ Stopword removal focuses attention on meaningful words, but can accidentally remove important words like "not."

✅ Stemming is fast but rough; lemmatization is slower but grammatically correct - the right choice depends on your task.

✅ Bag-of-Words counts words but ignores which words are actually distinctive.

✅ TF-IDF improves on Bag-of-Words by boosting words that are frequent in one document but rare across the whole corpus.

# Assignment

Submit a Jupyter Notebook (or the CSV file, as this lesson plan requests) containing:

1. Your own small dataset of at least 15 sentences (any topic).
2. A complete cleaning pipeline: tokenization, stopword removal, and either stemming or lemmatization (your choice, with a one-sentence justification for which you picked).
3. A Bag-of-Words table and a TF-IDF table for your cleaned dataset.
4. A half-page written explanation of why TF-IDF is generally more effective than raw word counts in NLP, using at least one concrete example from your own dataset.

# Final Reflection

1. In your own words, why can't a model use raw text directly?
2. What's the difference between what `.split()` does and what NLTK's `word_tokenize()` does?
3. Why might removing stopwords sometimes hurt a sentiment analysis task?
4. Give an example (not from today's lesson) of a word where stemming and lemmatization would produce different results.
5. In your own words, what does "Inverse Document Frequency" actually measure?
6. Why did we apply `clean_text()` before building our final TF-IDF matrix, rather than TF-IDF on the raw reviews directly?
7. What would happen to a word that appears in every single document, under TF-IDF, compared to under Bag-of-Words?

# Preview: Week 8 - Day 2

Tomorrow, we use the `cleaned_reviews.csv` file (and the TF-IDF techniques) from today to actually build a **sentiment classifier** - training a model to predict whether a review is positive or negative, using Logistic Regression and Naive Bayes, and evaluating it with the same accuracy/confusion-matrix tools from Week 5.